# Mirror Sem-Lex to Hugging Face Hub (Colab)

Use this when:
- Google Drive's anonymous-download quota is locking the Sem-Lex tarballs (~24h cooldown each time it gets exhausted), OR
- You don't have ~30 GB of free local disk to do `scripts/mirror_drive_to_hf.py` on your own machine.

Colab gives us **~100 GB ephemeral disk** + fast bandwidth + access to your authenticated Google Drive (which has a much higher per-user quota than the anonymous public-link quota that's been getting locked). Total runtime: ~30–60 min depending on Colab's network.

Run cells in order. The final cell prints the JSON snippet to paste into your `SEMLEX_DATA_URLS` GitHub Actions secret.

## Prereqs
1. **Hugging Face account** — https://huggingface.co/join (free)
2. **HF write token** — https://huggingface.co/settings/tokens → New token → role: write → copy `hf_…` value
3. **Empty HF dataset repo** — https://huggingface.co/new-dataset → owner = your username, name = `sem-lex-pilot-mirror` (or anything), public, license `apache-2.0`
4. **Sem-Lex Drive file IDs** — the same JSON you'd put in `SEMLEX_DRIVE_FILES` (only the 4 raw video/metadata files; the 3 `*-poses.tar.gz` are Req-7-forbidden and not mirrored)

**Important:** in your Google Drive web UI, the 4 Sem-Lex files must show up as accessible to you (either shared with your account or you opened them once via the share link). Mounting Drive in Colab makes them downloadable via the authenticated quota.

## 1. Install dependencies

In [ ]:
!pip install -q gdown huggingface_hub

## 2. Configure — paste your tokens / IDs here

Run the cell, then enter values at the prompts so they don't get saved into the notebook.

In [ ]:
import getpass, json

HF_TOKEN = getpass.getpass('HF write token (hf_…): ').strip()
HF_REPO = input('HF dataset repo (e.g. yourname/sem-lex-pilot-mirror): ').strip()

drive_json = input(
    'SEMLEX_DRIVE_FILES JSON (one line; 4 keys metadata/train/val/test → Drive file IDs):\n'
).strip()
DRIVE_FILES = json.loads(drive_json)
assert {'metadata', 'train', 'val', 'test'}.issubset(DRIVE_FILES), 'Need all 4 keys'
print(f'Got 4 file IDs. HF target: https://huggingface.co/datasets/{HF_REPO}')

## 3. Verify HF target repo exists + is writable
Fails fast if the repo doesn't exist or your token can't write to it.

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
info = api.repo_info(HF_REPO, repo_type='dataset')
print(f'Repo OK: {info.id}, private={info.private}')
# Write probe
import tempfile, os
with tempfile.NamedTemporaryFile(suffix='.txt', delete=False, mode='w') as f:
    f.write('mirror write probe\n')
    probe_path = f.name
api.upload_file(
    path_or_fileobj=probe_path,
    path_in_repo='.write_probe',
    repo_id=HF_REPO,
    repo_type='dataset',
    commit_message='write probe (delete me)',
)
os.unlink(probe_path)
print('Write probe OK — deleting it from the repo …')
api.delete_file(path_in_repo='.write_probe', repo_id=HF_REPO, repo_type='dataset')
print('Cleanup OK. Ready to mirror.')

## 4. Mirror each file (one at a time, peak disk = largest single file)

Roughly:
- metadata: ~13 MB, ~2 min
- val: ~few GB, ~5–10 min
- test: ~few GB, ~5–10 min
- train: ~23.7 GB, ~20–40 min

We delete each file after upload so the Colab disk doesn't fill. If anything fails partway, re-run this cell — it skips files already in the HF repo.

In [ ]:
import os, time, gdown
from huggingface_hub import HfApi

ROLE_TO_FILENAME = {
    'metadata': 'semlex_metadata.csv',
    'train':    'train.tar.gz',
    'val':      'val.tar.gz',
    'test':     'test.tar.gz',
}
WORK_DIR = '/content/semlex_mirror'
os.makedirs(WORK_DIR, exist_ok=True)
api = HfApi(token=HF_TOKEN)

existing = set(api.list_repo_files(HF_REPO, repo_type='dataset'))
final_urls = {}

for role, file_id in DRIVE_FILES.items():
    filename = ROLE_TO_FILENAME[role]
    repo_path = f'data/{filename}'
    url = f'https://huggingface.co/datasets/{HF_REPO}/resolve/main/{repo_path}'

    print(f'\n=== {role} ({filename}) ===')
    if repo_path in existing:
        print(f'  ⏭ Already in HF repo at {repo_path}, skipping')
        final_urls[role] = url
        continue

    local = os.path.join(WORK_DIR, filename)
    print(f'  Downloading from Drive: {file_id}')
    gdown.download(f'https://drive.google.com/uc?id={file_id}', local, quiet=False, resume=True, fuzzy=True)
    sz_gb = os.path.getsize(local) / 1e9
    print(f'  ✓ {sz_gb:.2f} GB on Colab disk')

    print(f'  Uploading to HF: {HF_REPO} → {repo_path}')
    t0 = time.monotonic()
    api.upload_file(
        path_or_fileobj=local,
        path_in_repo=repo_path,
        repo_id=HF_REPO,
        repo_type='dataset',
        commit_message=f'mirror {role} from Drive',
    )
    dur = time.monotonic() - t0
    print(f'  ✓ Uploaded in {dur:.0f}s ({(sz_gb*1000)/max(dur,1):.1f} MB/s)')

    os.remove(local)
    print(f'  Freed Colab disk: removed local {filename}')
    final_urls[role] = url

print('\n' + '=' * 70)
print('MIRROR COMPLETE — paste this into the SEMLEX_DATA_URLS GitHub secret:')
print('=' * 70)
print(json.dumps(final_urls, indent=2))

## 5. Add the secret + dispatch training

1. Copy the JSON above.
2. GitHub → repo → Settings → Secrets and variables → Actions → **New repository secret**.
3. Name: `SEMLEX_DATA_URLS`. Value: paste the JSON.
4. Save.
5. Go to Actions → Train Wave 1 → Run workflow, with `semlex_splits=train,val,test`. The fetcher will log `Using SEMLEX_DATA_URLS (direct HTTPS, recommended)` at the top and stream from your HF mirror — no quota, no fragility.

You can keep your old `SEMLEX_DRIVE_FILES` secret around as a fallback; `SEMLEX_DATA_URLS` takes priority.